# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mishellscripts/flyrank/blob/main/work/notebooks/capstone.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

Given limited editorial time, which declining pages
should be reviewed first? Can we predict recovery from a trained model and would it beat a hand-built rule?
A generated score consisting of decline severity and recovery potential is a mathematical approach that supports the decision-making process of content editors when manually checking content items.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

From FlyRank internship warehouse dataset: `fact_content_daily_performance` (daily impressions/clicks/position) and
`dim_content` (static content attributes). Cutoff T = 2026-02-01, chosen by
scanning every candidate month for the one maximizing clients with both
sufficient feature history and label runway (39 eligible clients).

Excluded: `trend_pct`/`trend_direction` as model features that are used only to define the decline gate itself.
 `is_deleted`, `is_published` are current-snapshot
fields with risk of reflecting post-T state. `last_optimized_date` used as a feature would risk leakage specifically because "was this page optimized" and "did this page recover" are plausibly the same event. `client_hash_id`/`content_hash_id` used only for joins and
GroupKFold groupings.

The data is anonymized with no client names, URLs, or private queries anywhere.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

For recovery prediction, both logistic regression and random forest were used, Both techniques were used with 5-fold cross valiation to compare performance (AUC/precision). With a small dataset, it was important to use cross-validation to lower overfitting risk to a randomly chosen client set.

Every feature listed in 2 were checked before inclusion. Furthermore, the cross-validated folds did not contain the same clients. The recovery label windows and feature windows never overlapped. The label `recovered_by_T1 = 1` represents an increase in impressions from one 30d period to the next (both after T). This was the result used to compare with the prediction model for precision.

Feature selection methods used were PCA and Elastic Net. PCA [...]. Elastic Net [...]. Elastic Net was selected as the final feature selection method because it handled correlated predictors
directly with similar performance without needing PCA's feature-blending complexity. For the final feature set, the random forest max depth was tuned via GroupKFold-aware grid search.

For scoring, the baseline and final model share the same
formula: impact_at_risk * unlikeliness-to-recover. The baseline used avg_position as an
untested proxy for recovery likelihood. The final model replaces that proxy with a probability of recovery, a cross-validated
prediction from real historical outcomes.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

| | Full features | Elastic Net (final) |
|---|:---:|:---:|
| AUC (random forest) | 0.692–0.708 | 0.706–0.711 |
| Precision@20 | 0.890 | 0.940 |
| Precision@50 | 0.892–0.912 | 0.880 |

Base rate varies substantially across folds/clients (0.13–0.87), so raw
precision figures are read alongside lift-over-base-rate, not in isolation
— a raw 0.90 precision in a fold with an 0.85 base rate reflects little
model skill; the same 0.90 in a fold with a 0.20 base rate reflects real
discrimination.

Error analysis: 28.2% of held-out predictions wrong. Five actual wrong
cases (all predicted "stayed broken," all recovered) shared two patterns:
all tagged model_used=gpt-4o-mini (consistent with the confound cluster
above resurfacing as a visible error pattern), and all had thin visibility
profiles — small pages the model may weight too heavily on visibility
signals that are least reliable for the smallest content.

## 5. Limitations

*What this work cannot claim.*

This queue answers "which known problems deserve attention first," not
"which healthy pages are about to decline" — a different, complementary
model, not built here. No causal claim is made that reviewing a flagged
page causes recovery — this analysis is observational, no controlled
comparison exists between refreshed and non-refreshed pages.

days_since_last_update's strong, consistent coefficient likely reflects
editorial attention arriving during/after a decline, not pre-existing
staleness — content_updated_date falls after T for ~91% of rows, since
this table is a live snapshot with no point-in-time history.

content_type's relevance to recovery is unconfirmed — recovered by manual
removal and RFECV, not by Elastic Net, forward selection, Lasso, or PCA.
Reported as directional, not established.

Model built on only 38–39 eligible clients; scores for very new or
thin-history clients should be treated as lower confidence.

Calibration check shows predicted P(recovery) is systematically overstated
across most of the range — priority scores likely understate urgency for
some mid-probability pages. Ranking order (AUC/precision@K) is unaffected;
absolute probability values should be read cautiously.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.